# Waymo trajectory Transformer: Colab runner
Runtime → Change runtime type → **T4 GPU**. Put `waymo-trajectory-transformer.zip` in Google Drive under `MyDrive/waymo_project/` first.

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()          # sign in with your Waymo-registered Google account
drive.mount('/content/drive')

In [ ]:
!unzip -q -o /content/drive/MyDrive/waymo_project/waymo-trajectory-transformer.zip -d /content
%cd /content/waymo-trajectory-transformer
!python tests/test_pipeline.py

## Settings
`N_TRAIN` = number of Waymo training files (max 1000). Free Colab handles about 100 comfortably.

In [ ]:
N_TRAIN = 100
EPOCHS = 30 if N_TRAIN <= 100 else 10
DATA = f"data_{N_TRAIN}"
RUNS = f"/content/drive/MyDrive/waymo_project/runs_{N_TRAIN}"   # saved to Drive

In [ ]:
# download + convert (raw files deleted as it goes; safe to re-run)
!python fetch_convert.py --split validation --start 0 --end 5  --out {DATA}/val
!python fetch_convert.py --split validation --start 5 --end 10 --out {DATA}/test
!python fetch_convert.py --split training   --start 0 --end {N_TRAIN} --out {DATA}/train --batch 10
!cp -r {DATA} /content/drive/MyDrive/waymo_project/

In [ ]:
!python train.py --data {DATA} --eval_cv
!python train.py --data {DATA} --K 1 --epochs {EPOCHS} --out {RUNS}
!python train.py --data {DATA} --K 3 --epochs {EPOCHS} --out {RUNS}

In [ ]:
!python analyze.py --data {DATA} --runs {RUNS} --split test
from IPython.display import Image, display
for f in ["fig1_single_overlays", "fig2_multimodal", "fig3_error_bars", "fig4_loss_curves"]:
    display(Image(f"{RUNS}/analysis_test/{f}.png"))